# Day 23 · Calibrate an LLM judge against your own labels

**First-principles question:** When a model grades your agent, how often does it agree with you?

The full task (learn → build → record) is in this folder's `README.md`. This notebook is your workbench:
run cells top to bottom with **Shift + Enter**, fill in every cell marked **TODO**, and finish with the
**✅ self-check** cell at the end.

In [ ]:
import sys
from pathlib import Path

def find_root():
    """The fde-first-principles folder (it has progress.json)."""
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "progress.json").exists():
            return p
    raise RuntimeError("Open this notebook from inside your fde-first-principles folder.")

ROOT = find_root()
sys.path.insert(0, str(ROOT))  # so `import lab.…` works

import csv
import numpy as np
import pandas as pd
from lab.acme import DATA, load_cases, load_results
from lab.llm import chat, extract_json

cases = {c["id"]: c for c in load_cases()}
POLICY = (DATA / "policy.md").read_text(encoding="utf-8")
LABELS = Path("labels.csv")

## 1 · Your labels
The cell writes `labels.csv` with 40 replies from B and C. Open it, and for each row fill in `you` (pass or fail) and a one-line `critique`. Then re-run the next cell.

In [ ]:
if not LABELS.exists():
    rows = [("B", r) for r in load_results("B")] + [("C", r) for r in load_results("C")]
    rng = np.random.default_rng(7)
    pick = [rows[i] for i in rng.choice(len(rows), size=min(40, len(rows)), replace=False)]
    with LABELS.open("w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["version", "id", "reply", "you", "critique"])
        for version, r in pick:
            w.writerow([version, r["id"], r.get("reply", ""), "", ""])
labels = pd.read_csv(LABELS).fillna("")
print(labels["you"].value_counts())

## 2 · The judge
Critique first, then the verdict. Add three of your labelled rows as examples.

In [ ]:
JUDGE = """You grade replies from Acme Fibre's support assistant.
Policy:
{policy}

Question: does the reply really solve the customer's problem, politely, within the policy?
Write a one-line critique, then a verdict. Answer as JSON: {{"critique": "...", "verdict": "pass" or "fail"}}
"""
# TODO: add three labelled examples (email, reply, your critique, your verdict) to JUDGE

def judge(case, reply, order="policy-first"):
    system = JUDGE.format(policy=POLICY)
    user = f"Email: {case['message']}\n\nReply: {reply}"
    if order == "reply-first":
        system, user = "You grade support replies. Answer as JSON.", user + "\n\n" + system
    out = chat([{"role": "system", "content": system}, {"role": "user", "content": user}], temperature=0)
    return extract_json(out["content"]).get("verdict", "fail")

labels["judge"] = [judge(cases[r.id], r.reply) for r in labels.itertuples()]

## 3 · Agreement on the held-out half

In [ ]:
held_out = labels.iloc[len(labels) // 2:]

def kappa(a, b):
    """Cohen's kappa for two lists of 'pass'/'fail'."""
    # TODO: observed = share where a == b
    #       chance = P(a pass) * P(b pass) + P(a fail) * P(b fail)
    #       return (observed - chance) / (1 - chance)
    ...

display(pd.crosstab(held_out["you"], held_out["judge"], rownames=["you"], colnames=["judge"]))
print(f"kappa on the held-out half: {kappa(list(held_out['you']), list(held_out['judge'])):.2f}")

## 4 · Two bias checks

In [ ]:
sample = labels.iloc[:10]
flipped = sum(judge(cases[r.id], r.reply) != judge(cases[r.id], r.reply, order="reply-first") for r in sample.itertuples())
good = labels[labels["you"] == "pass"].iloc[0]
longer = judge(cases[good.id], good.reply + " " + good.reply)
print(f"order swap changed {flipped} of {len(sample)} verdicts · doubled good reply → {longer}")

**Would you let this judge grade without you? Why, with kappa?** …

In [ ]:
# ✅ Self-check: run me last. A clear message tells you what to fix.
assert (labels["you"].isin(["pass", "fail"])).sum() >= 20, "Step 1: label at least 20 rows in labels.csv"
assert kappa(["pass", "fail", "pass", "fail"], ["pass", "fail", "pass", "fail"]) == 1
assert abs(kappa(["pass", "pass", "fail", "fail"], ["pass", "fail", "pass", "fail"])) < 1e-9, "Chance agreement gives kappa 0"
print("All checks passed ✅  Now write 'In my words' in README.md and record your video.")